## Exploratory data anaylsis of CyCIF dataset


In [ ]:
import os
import sys
import gc
import cv2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import tifffile
import xml.etree.ElementTree as ET

from skimage.transform import rescale
from IPython.display import display

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
sys.path.append('..')
import IO, zonation

### CyCIF multiplexed image registration

#### Full 3D sample

| Cycle | Opal 520 | Opal 690 | Opal 570 | Opal 780 | 
| --- | --- | --- | --- | --- |
| 1 | B-Catenin | ASS1 | GS | CYP3A4 |
| 2 | Pan-CK | CD31 | Collagen | NaN |
| 3 | CD45 | CD68 | Arg1 | Lyve1 |
| 4 | CD56 | Vimentin | PU1 | CD3 |

(1). Load Cy-IF images from google cloud bucket

In [ ]:
CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
PROJECT_ID = 'azizilab-cell-segmentation'
BUCKET_ID = 'liver_3d'
HOME_PATH = 'CyIF/qupath/'

cyif_reader = IO.CyIFGcloudReader(CREDENTIAL_PATH,
                                  PROJECT_ID,
                                  BUCKET_ID,
                                  HOME_PATH,
                                  scale=1,
                                  sigma=10)


(2). Within-cycle registration

In [ ]:
out_path = '../data/cycif/raw/'

for slide_id in cyif_reader.slide_ids:
    annot_imgs = cyif_reader.load_imgs(slide_id,
                                       chans_to_ignore={'Opal 620'},
                                       verbose=True)    
    
    annot_imgs_warped = cyif_reader.register_cycles(annot_imgs, 
                                                    verbose=True)
    
    cyif_reader.save_annotated_imgs(annot_imgs_warped, 
                                    output_path=out_path)

    gc.collect()
    # del annot_imgs, annot_imgs_warped

### 2D/3D zonation

- TODO: **Figure out misalignment issues**, currently only works for Slide 1 & 3
- Selected zonation markers: `GS 647`, `CYP3A4`, `ASS1 PE` (Round 1), `Col I`, `Pan CK` (Round 2)
- Try construct 3D graphs

In [ ]:
from skimage.filters import gaussian, sobel, threshold_otsu, threshold_minimum
from skimage.exposure import rescale_intensity
from skimage.registration import optical_flow_ilk
from scipy import ndimage as ndi

In [ ]:
from importlib import reload
reload(IO)

In [ ]:
def remove_holes(roi, min_area):
    """
    Remove holes & FP lslands in binary ROI mask
    """
    roi_filtered = roi.copy().astype(np.uint8)
    roi_labeled, n_features = ndi.label(roi)
    
    for i in range(1, n_features+1):
        if (roi_labeled == i).sum() < min_area:
            roi_filtered[roi_labeled == i] = 0
            
    return ndi.binary_fill_holes(roi_filtered).astype(np.uint8)


def get_roi_mask(dapi, sigma=10, min_area=None):
    """
    Compute slice foreground / background mask
    """
    if min_area is None:
        # Defaulting min_area as 1/4 of the whole FOV area
        min_area = dapi.shape[0]//2*dapi.shape[1]//2
        
    dapi_smoothed = rescale_intensity(gaussian(dapi, sigma=10), out_range=(0, 1))
    threshold = threshold_otsu(dapi_smoothed)
    return remove_holes(dapi_smoothed > threshold, min_area=min_area)
    

Load aligned CyIF tiffs:

In [ ]:
data_path = '../data/cycif/aligned/registered_slides/'
cyif_imgs, filenames = IO.load_annot_tiffs(data_path, ext='ome.tiff')

# TODO: rerun on full z-slices after solving the misalignment issues
cyif_imgs, filenames = cyif_imgs[:7], filenames[:7]
filenames

In [ ]:
chan_labels = cyif_imgs[0].keys()
print(list(chan_labels))

#### Examine continuity of zonation markers:

In [ ]:
# Compute ROI (foreground) mask  + AF mask for each z-slice
# roi_masks = [get_roi_mask(img['DAPI'])
#              for img in cyif_imgs]

af1_masks = [(img['Sample AF_01'] > 40).astype(np.uint8)
             for img in cyif_imgs]
af2_masks = [(img['Sample AF_02'] > 40).astype(np.uint8)
             for img in cyif_imgs]

In [ ]:
# Try subtracting w/ AF within ROI foregrond regions
# Normalize each channel to [0, 1]

zone_labels = ['GS 647', 'CYP3A4', 'ASS1 PE', 'Col I', 'Pan CK']
zonation_chans = {}

for label in zone_labels:
    chans = []
    for i, (img, roi) in enumerate(zip(cyif_imgs, roi_masks)):
        af = af1_masks[i] if label in zone_labels[:3] else af2_masks[i]
        
        chan = rescale_intensity(img[label], out_range=(0, 1)) - af
        chan[chan < 0] = 0
        chan[roi == 0] = 0
        chans.append(chan)

        # chan = rescale_intensity(img[label].copy(), out_range=(0, 1))
        # chan[roi == 0] = 0
        # chans.append(chan)
        
    zonation_chans[label] = np.array(chans)

del label, chans, img, roi, chan, af
# del img, roi, chan

In [ ]:
def disp_chans(img, title=None, ncols=4):
    depth = len(img)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(img[idx])
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
# Denoise w/ binarized AF
for label in zone_labels:
    disp_chans(zonation_chans[label], title=label)
del label

In [ ]:
# Save sample overlayed zonation markers
for label, chan in zonation_chans.items():
    tifffile.imwrite('../data/cycif/backup/CyIF_{}_slices.tif'.format(label), chan, metadata={'axes': 'CYX'})

In [ ]:
def disp_optical_flow(z1, z2, title=None):
    # Reference: 
    # https://scikit-image.org/docs/stable/auto_examples/registration/plot_opticalflow.html
    v, u = optical_flow_ilk(z1, z2)
    norm = np.sqrt(u** + v**2)
    nvec = 20
    nl, nc = z1.shape
    step = max(nl//nvec, nc//nvec)
    
    y, x = np.mgrid[:nl:step, :nc:step]
    u_ = u[::step, ::step]
    v_ = v[::step, ::step]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))
    ax1.imshow(z1)
    ax1.set_title('Slice z')

    ax2.imshow(z2)
    ax2.set_title('Slice z+1')
    
    ax3.imshow(norm)
    ax3.quiver(x, y, u_, v_, color='r', units='dots',
               angles='xy', scale=0.8, scale_units='xy', lw=2)
    ax3.set_title("Optical flow vector field")

    plt.tight_layout()
    plt.suptitle(title, y=1.1)
    plt.show()

In [ ]:
for label in zone_labels[:3]:
    print('Channel', label)
    print('==='*5)
    for i in range(len(zonation_chans[label])-1):
        z1, z2 = zonation_chans[label][i], zonation_chans[label][i+1]
        disp_optical_flow(z1, z2, title='Transition of {0} slices({1} vs. {2})'.format(label, i, i+1))   
    print('\n\n\n')

del z1, z2

#### 3D heat diffusion 
- Trajectory origin: `ASS` for now; destination `GS`
- 6-connected components as the graph unit
- **Todo:** add anisotropy scaling along Z-axis

**(1)**. Construct 3D ROI & CV / PV tentative mask

**TODO**: expand # Z-slices & increase weights along in-plane (Z-axis) edges for anisotropy

In [ ]:
# CV / PV mask w/ otsu threshold

# roi = np.array(roi_masks)
# cv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['GS 647']):
#     threshold = threshold_otsu(chan)
#     cv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# pv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['ASS1 PE']):
#     threshold = threshold_otsu(chan)
#     pv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# mask = np.zeros_like(roi, dtype=np.int32)
# mask[np.logical_and(pv_mask == 1, cv_mask == 0)] = -1
# mask[np.logical_and(pv_mask == 0, cv_mask == 1)] = 1

# Dilate on mask
# for i, mask_slc in enumerate(mask):
#     cv_mask = binary_dilation(mask_slc == 1, np.ones((5, 5))).astype(np.uint8)
#     pv_mask = binary_dilation(mask_slc == -1, np.ones((5, 5))).astype(np.uint8)

#     mask[i][pv_mask == 1] = -1
#     mask[i][cv_mask == 1] = 1

# del cv_mask, pv_mask

# for i, (roi_slc, mask_slc) in enumerate(zip(roi, mask)):
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
#     ax1.imshow(roi_slc)
#     ax1.set_title('Tissue ROI (z={})'.format(i))
#     ax1.axis('off')
    
#     ax2.imshow(mask_slc, cmap='coolwarm')
#     ax2.set_title('CV/PV mask (z={})'.format(i))
#     ax2.axis('off')
    
#     plt.tight_layout()
#     plt.show()

# del roi_slc, mask_slc

# # Save current ROI & CV/PV mask
# tifffile.imwrite('../data/cycif/backup/CyIF_ROI.tif', roi)
# tifffile.imwrite('../data/cycif/backup/CyIF_zone_masks.tif', mask)

**(2)** Construct 3D graph & run diffusion inference

In [ ]:
# 3D heat diffusion model

# Load computed roi & mask
# DEBUG: 3D diffusion's correction w/ Z-slices 8-11
roi = tifffile.imread('../data/cycif/backup/CyIF_ROI.tif')[-4:]
mask = tifffile.imread('../data/cycif/backup/CyIF_zone_masks.tif')[-4:]


G = create_graph(roi=roi)
add_graph_props(G, cv_indices, pv_indices, depth=roi.shape[0])
u_i, interior_nodes = compute_interior_temp(G)
u = assign_diffusion_temp(u_i, 
                          interior_nodes,
                          cv_coords=np.where(mask == 1),
                          pv_coords=np.where(mask == -1),
                          shape=mask.shape)

In [ ]:
t0 = time.perf_counter()

model = zonation.HeatDiffusion(vein_prior=mask,
                               roi=roi,
                               ndim=3)
U_i, interior_nodes = model.get_interior_U()
U = model.infer_zone_dynamics()
# lobule_layers = model.infer_zones()

t1 = time.perf_counter()
print('Heat diffusion for dim {0} x {1} x {2} takes {2}s'.format(mask.shape[0], mask.shape[1], mask.shape[2], t1-t0))
del U_i, interior_nodes


In [ ]:
for i, u_slc in enumerate(U):
    plt.figure()
    plt.imshow(u_slc, cmap='coolwarm')
    plt.title('Diffused dynamics (z={})'.format(i))
    plt.show()

In [ ]:
# Save inferred heat dynamics
tifffile.imwrite('../data/cycif/backup/CyIF_zone_dynamics.ome.tif', u, metadata={'axes': 'ZYX'})

In [ ]:
# Visualize aligned CyIF


---